# Tier 2 — LSTM Autoencoder for Unsupervised Anomaly Detection

**This notebook is the scaling path. Nothing in it is deployed to the microcontroller.**

## What the deployed system cannot do

The random forest running on the Pico W is a *supervised* classifier. It was shown
labelled examples of three states and learned to tell them apart. That is a good fit
for the problem as posed, and it converts to a few kB of `if`/`else` that a Cortex-M0+
can evaluate in microseconds.

But it has a hard limit that is easy to talk past: **it can only recognise faults
that resemble its training data.** Ask it about a rotor bar fracture, a loose
foundation bolt, a lubrication failure, or a bearing defect on the *inner* race
rather than the outer, and it will not say "I don't know". It has three output
slots and it will confidently pick one. A model with no capacity to express
uncertainty about novel inputs is a genuine liability on a machine someone depends on.

That is a problem you cannot solve by collecting more labels, because the failures
that matter most are the ones nobody has catalogued yet.

## The reframing

Stop asking *"which of these three faults is this?"* and start asking
*"does this look like the machine I know?"*

Train a model to **reconstruct healthy behaviour only**. Feed it a window and
measure how badly it fails to reproduce it. That reconstruction error becomes an
anomaly score:

- a healthy machine reconstructs well → low error
- *anything* unfamiliar reconstructs badly → high error

Crucially, the model never has to have seen a fault. It only has to know
normality well. This is why unsupervised anomaly detection is what production PdM
platforms actually run — healthy data is abundant and free, while labelled
run-to-failure data is rare and expensive.

## Why an LSTM specifically

Degradation is a **process over time**, not a property of a single 256 ms window.
A feedforward autoencoder sees each window independently and so cannot notice that
kurtosis has been creeping up for the last forty minutes. An LSTM consumes a
*sequence* of feature vectors and carries state between them, so it can model the
temporal texture of normal operation — including normal transients like a load
change, which a per-window model would flag as anomalies.

That same sequence modelling is the foundation of real Remaining Useful Life
estimation: once you can model trajectories, you can extrapolate them.

## Setup

Requires PyTorch, which is **not** in the core `requirements.txt` on purpose — the
deployed model has no dependency on it.

```
py -m pip install -r requirements.txt -r requirements-tier2.txt
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

from feature_spec import FEATURE_NAMES, N_FEATURES, WINDOW_N
from extract_features import extract_features

# The continuous-severity machine model. This is the same physics the firmware's
# Wokwi shim and the demo validator use -- a motor whose fault severity we can
# dial smoothly from 0 (healthy) to 1 (critical), rather than the three discrete
# classes the supervised dataset is built from.
from demo_sweep_check import firmware_window as machine_window

torch.manual_seed(0)
rng = np.random.default_rng(11)

# Plot styling, matching the rest of the project.
INK, INK_MUTED, GRID, SURFACE = "#0b0b0b", "#898781", "#e1e0d9", "#fcfcfb"
SERIES_BLUE, STATUS_CRITICAL = "#2a78d6", "#e34948"

def style(ax):
    ax.set_facecolor(SURFACE)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    for s in ("left", "bottom"):
        ax.spines[s].set_color("#c3c2b7"); ax.spines[s].set_linewidth(0.8)
    ax.tick_params(colors=INK_MUTED, labelsize=9, length=3, width=0.8)
    ax.grid(color=GRID, linewidth=0.8); ax.set_axisbelow(True)

print("torch", torch.__version__, "| features", N_FEATURES)

## 1. Run-to-failure trajectories

The supervised dataset is a pile of independent windows with class labels. A
sequence model needs something different: **runs**. Each run follows one machine
from healthy through to failure, sampled every few minutes.

`severity` ramps from 0 toward 1 over the life of the run. The ramp is not linear —
real degradation accelerates, because damage creates the conditions for more damage
(a spall roughens the race, which increases impact energy, which widens the spall).
An exponential-ish profile captures that, and it is also what makes RUL prediction
non-trivial: early in life the trend is almost flat.

We also give each run its own random rate and a per-window jitter, so the model
cannot memorise one canonical curve.

In [ ]:
N_RUNS       = 48     # independent machines
RUN_LENGTH   = 70     # windows per run (think: one reading every few minutes)
CURRENT_BASE = 12.0
TEMP_BASE    = 45.0

def degradation_profile(n, rate, rng):
    \"\"\"Severity 0 -> ~1 over a run, accelerating, with realistic measurement jitter.\"\"\"
    t = np.linspace(0.0, 1.0, n)
    # Accelerating damage: slow, almost-flat start then a sharp knee.
    sev = (np.exp(rate * t) - 1.0) / (np.exp(rate) - 1.0)
    sev = np.clip(sev + rng.normal(0.0, 0.015, n), 0.0, 1.0)
    return sev

def build_run(rng):
    rate = rng.uniform(2.0, 4.5)
    sev = degradation_profile(RUN_LENGTH, rate, rng)
    rows = []
    for s in sev:
        ax, ay, az = machine_window(float(s), rng)
        # Current and temperature track severity, with their own noise.
        cur = CURRENT_BASE + 13.0 * s + rng.normal(0.0, 0.4)
        tmp = TEMP_BASE + 55.0 * s + rng.normal(0.0, 1.2)
        rows.append(extract_features(ax, ay, az,
                                     np.full(WINDOW_N, cur),
                                     np.full(WINDOW_N, tmp)))
    return np.asarray(rows), sev

runs, severities = [], []
for i in range(N_RUNS):
    x, s = build_run(rng)
    runs.append(x); severities.append(s)
    if (i + 1) % 12 == 0:
        print(f"  generated {i + 1}/{N_RUNS} runs")

runs = np.asarray(runs)              # (N_RUNS, RUN_LENGTH, N_FEATURES)
severities = np.asarray(severities)
print("runs:", runs.shape)

In [ ]:
# What a few runs look like. Kurtosis is plotted because it is the feature the
# supervised model leans on -- note how noisy it is window to window, which is
# exactly why a sequence model has something to add over a per-window threshold.
fig, ax = plt.subplots(figsize=(7.4, 4.2), facecolor=SURFACE)
ki = FEATURE_NAMES.index("vib_kurtosis")
for r in range(6):
    ax.plot(runs[r, :, ki], linewidth=1.4, alpha=0.75, color=SERIES_BLUE)
ax.set_xlabel("window index (machine age)", fontsize=9.5, color=INK, labelpad=8)
ax.set_ylabel("vib_kurtosis", fontsize=9.5, color=INK, labelpad=8)
ax.set_title("Six independent run-to-failure trajectories",
             fontsize=11, color=INK, pad=12, loc="left")
style(ax)
plt.tight_layout(); plt.show()

## 2. Train on healthy data only

This is the part that makes the method unsupervised, and the part that is easiest
to get subtly wrong.

We take only the **first 25% of each run** as "known healthy". Two consequences
worth being explicit about:

1. **No fault labels are used anywhere in training.** The model never sees a
   CRITICAL window, never sees a class index. It only learns what normal looks like.
2. **The normaliser must be fitted on healthy data only.** If you fit the scaler
   across the whole dataset, the mean and standard deviation absorb information
   about the faults — the anomalies get scaled toward the centre and the model's
   job becomes artificially easy. This is a textbook leakage bug in anomaly
   detection and it produces beautiful, meaningless results.

In [ ]:
HEALTHY_FRACTION = 0.25
SEQ_LEN = 10          # windows of context the LSTM sees at once

n_healthy = int(RUN_LENGTH * HEALTHY_FRACTION)
healthy_windows = runs[:, :n_healthy, :].reshape(-1, N_FEATURES)

# Fit the scaler on HEALTHY DATA ONLY. See the note above.
mu = healthy_windows.mean(axis=0)
sigma = healthy_windows.std(axis=0)
sigma[sigma < 1e-9] = 1.0

def normalise(x):
    return (x - mu) / sigma

def to_sequences(arr2d, seq_len=SEQ_LEN):
    \"\"\"Sliding windows over one run: (T, F) -> (T-seq_len+1, seq_len, F).\"\"\"
    return np.stack([arr2d[i:i + seq_len]
                     for i in range(len(arr2d) - seq_len + 1)])

train_seqs = np.concatenate([
    to_sequences(normalise(runs[r, :n_healthy, :])) for r in range(N_RUNS)
])
rng.shuffle(train_seqs)
split = int(0.85 * len(train_seqs))
train_x, val_x = train_seqs[:split], train_seqs[split:]

print(f"healthy windows per run : {n_healthy}")
print(f"training sequences      : {train_x.shape}")
print(f"validation sequences    : {val_x.shape}")

## 3. The autoencoder

An encoder LSTM compresses a 10-window sequence down to an 8-number latent vector;
a decoder LSTM tries to rebuild the original 10 windows from that alone.

The **bottleneck is the whole mechanism**. Eight numbers is far too few to store a
sequence verbatim, so the network is forced to learn the *structure* of normal
operation — which features move together, how fast they drift, what a normal load
change looks like. Feed it something structurally unfamiliar and the compression
throws away the wrong information, so the reconstruction comes back wrong.

Make the latent space too large and the model learns a near-identity function:
it will reconstruct faults beautifully too, and the anomaly score goes flat. That
failure mode is silent, which is why the diagnostic plot in section 5 matters.

In [ ]:
class LSTMAutoencoder(nn.Module):
    def __init__(self, n_features, hidden=32, latent=8, seq_len=SEQ_LEN):
        super().__init__()
        self.seq_len = seq_len
        self.encoder = nn.LSTM(n_features, hidden, batch_first=True)
        self.to_latent = nn.Linear(hidden, latent)
        self.from_latent = nn.Linear(latent, hidden)
        self.decoder = nn.LSTM(hidden, hidden, batch_first=True)
        self.output = nn.Linear(hidden, n_features)

    def forward(self, x):
        _, (h, _) = self.encoder(x)          # h: (1, B, hidden)
        z = self.to_latent(h[-1])            # (B, latent)  <- the bottleneck
        seed = self.from_latent(z)           # (B, hidden)
        repeated = seed.unsqueeze(1).repeat(1, self.seq_len, 1)
        decoded, _ = self.decoder(repeated)
        return self.output(decoded)

model = LSTMAutoencoder(N_FEATURES)
print(model)
print("parameters:", sum(p.numel() for p in model.parameters()))

In [ ]:
EPOCHS, BATCH = 60, 64

train_t = torch.tensor(train_x, dtype=torch.float32)
val_t = torch.tensor(val_x, dtype=torch.float32)

opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()
history = []

for epoch in range(EPOCHS):
    model.train()
    perm = torch.randperm(len(train_t))
    running = 0.0
    for i in range(0, len(train_t), BATCH):
        batch = train_t[perm[i:i + BATCH]]
        opt.zero_grad()
        loss = loss_fn(model(batch), batch)
        loss.backward()
        opt.step()
        running += loss.item() * len(batch)

    model.eval()
    with torch.no_grad():
        val_loss = loss_fn(model(val_t), val_t).item()
    history.append((running / len(train_t), val_loss))

    if (epoch + 1) % 10 == 0:
        print(f"  epoch {epoch + 1:3d}  train {history[-1][0]:.5f}  "
              f"val {val_loss:.5f}")

In [ ]:
hist = np.asarray(history)
fig, ax = plt.subplots(figsize=(7.0, 3.8), facecolor=SURFACE)
ax.plot(hist[:, 0], color=SERIES_BLUE, linewidth=2, label="train")
ax.plot(hist[:, 1], color=INK_MUTED, linewidth=2, linestyle=(0, (4, 3)),
        label="validation")
ax.set_xlabel("epoch", fontsize=9.5, color=INK, labelpad=8)
ax.set_ylabel("reconstruction MSE", fontsize=9.5, color=INK, labelpad=8)
ax.set_title("Training on healthy data only", fontsize=11, color=INK,
             pad=12, loc="left")
ax.legend(frameon=False, fontsize=9, labelcolor=INK)
style(ax)
plt.tight_layout(); plt.show()

## 4. Choosing the threshold

The model outputs a continuous error, but an alarm needs a boolean. Where the line
goes is a **business decision disguised as a statistic**, and it should be made
explicitly rather than by picking a round number.

We set it at the 99th percentile of reconstruction error on *healthy validation
data*. That is a direct statement of tolerance: roughly 1 in 100 healthy windows
will trip the alarm.

Whether that is right depends entirely on cost asymmetry. If a false alarm means a
technician walks over and looks at a machine, 1% is cheap. If it means stopping a
production line, 1% is ruinous and you want 99.9% — accepting that you will detect
faults later. A node reporting every 256 ms sees ~14,000 windows an hour, so a 1%
false-positive rate is 140 alarms an hour; in practice you would also require N
consecutive exceedances before annunciating, which trades a little latency for an
enormous reduction in nuisance trips.

In [ ]:
def reconstruction_error(seqs):
    \"\"\"Per-sequence mean squared reconstruction error.\"\"\"
    model.eval()
    with torch.no_grad():
        t = torch.tensor(seqs, dtype=torch.float32)
        recon = model(t)
        return ((recon - t) ** 2).mean(dim=(1, 2)).numpy()

val_errors = reconstruction_error(val_x)
THRESHOLD = float(np.percentile(val_errors, 99))

print(f"healthy validation error: median {np.median(val_errors):.5f}, "
      f"max {val_errors.max():.5f}")
print(f"alarm threshold (p99)   : {THRESHOLD:.5f}")

## 5. Does it detect degradation it was never shown?

Now run the full trajectories — including the critical end that the model has never
seen in any form — and watch the anomaly score.

In [ ]:
detect_points, knee_points = [], []
fig, ax = plt.subplots(figsize=(7.6, 4.6), facecolor=SURFACE)

for r in range(N_RUNS):
    seqs = to_sequences(normalise(runs[r]))
    err = reconstruction_error(seqs)
    if r < 10:
        ax.plot(np.arange(len(err)) + SEQ_LEN - 1, err,
                color=SERIES_BLUE, linewidth=1.3, alpha=0.6)
    over = np.where(err > THRESHOLD)[0]
    if len(over):
        detect_points.append(over[0] + SEQ_LEN - 1)
        knee_points.append(int(np.argmax(severities[r] > 0.5)))

ax.axhline(THRESHOLD, color=STATUS_CRITICAL, linewidth=1.6,
           linestyle=(0, (5, 3)))
ax.text(1, THRESHOLD * 1.08, "alarm threshold (p99 of healthy)",
        fontsize=8.5, color=STATUS_CRITICAL)
ax.set_yscale("log")
ax.set_xlabel("window index (machine age)", fontsize=9.5, color=INK, labelpad=8)
ax.set_ylabel("reconstruction error (log)", fontsize=9.5, color=INK, labelpad=8)
ax.set_title("Anomaly score rises long before failure",
             fontsize=11, color=INK, pad=12, loc="left")
style(ax)
plt.tight_layout(); plt.show()

print(f"runs that alarmed      : {len(detect_points)}/{N_RUNS}")
print(f"median detection window: {np.median(detect_points):.0f} of {RUN_LENGTH}")
print(f"median severity-50% at : {np.median(knee_points):.0f}")
print(f"=> median warning lead : "
      f"{np.median(knee_points) - np.median(detect_points):.0f} windows "
      f"before the machine is half-degraded")

## 6. The real test: a fault the system has never encountered

Everything so far still used the same outer-race defect the supervised forest
knows about. The actual claim is detection of *novel* failure modes, so here is one
the system has genuinely never seen in any form:

**an inner-race defect combined with severe shaft imbalance.** The inner race
rotates with the shaft, so its defect frequency (BPFI, here ~136 Hz) differs from
BPFO, and the impacts are modulated much more deeply because the defect passes
through the load zone once per revolution. Add a large 1x imbalance component and
the spectral signature is nothing like the training distribution.

The supervised forest has three output slots and none of them is "unknown fault",
so it must pick one. The autoencoder has no such constraint.

In [ ]:
from feature_spec import FS_HZ, SHAFT_HZ, IMPACT_TAU_S

BPFI_HZ = 136.0        # inner race -- a frequency nothing in this project has seen

def novel_fault_window(rng, imbalance=0.22, impact=0.35):
    t = np.arange(WINDOW_N) / FS_HZ
    # Severe 1x imbalance -- dominant, and far larger than any training sample.
    v = imbalance * np.sin(2 * np.pi * SHAFT_HZ * t + rng.uniform(0, 6.28))
    # Deeply modulated inner-race impacts (defect rotates through the load zone).
    env = 1.0 + 0.9 * np.sin(2 * np.pi * SHAFT_HZ * t)
    period = 1.0 / BPFI_HZ
    k = -1
    while k * period < WINDOW_N / FS_HZ:
        dt = t - k * period * (1.0 + rng.normal(0, 0.012))
        live = dt >= 0
        v[live] += (impact * env[live] * np.exp(-dt[live] / IMPACT_TAU_S)
                    * np.sin(2 * np.pi * 420.0 * dt[live]))
        k += 1
    v += rng.normal(0, 0.02, WINDOW_N)
    ax = np.clip(0.94 * v, -2, 2)
    ay = np.clip(0.30 * v, -2, 2)
    az = np.clip(1.0 + 0.16 * v, -2, 2)
    return ax, ay, az

novel = []
for _ in range(SEQ_LEN * 4):
    a, b, c = novel_fault_window(rng)
    novel.append(extract_features(a, b, c,
                                  np.full(WINDOW_N, 13.0),    # current: normal
                                  np.full(WINDOW_N, 50.0)))   # temp: normal
novel = np.asarray(novel)

novel_err = reconstruction_error(to_sequences(normalise(novel)))
print(f"healthy median error : {np.median(val_errors):.5f}")
print(f"threshold            : {THRESHOLD:.5f}")
print(f"novel-fault median   : {np.median(novel_err):.5f}  "
      f"({np.median(novel_err) / THRESHOLD:.1f}x threshold)")
print(f"flagged              : {100 * np.mean(novel_err > THRESHOLD):.0f}% of windows")

In [ ]:
# What does the supervised forest say about the same data?
import joblib, os
forest = joblib.load(os.path.join("artifacts", "rf_model.joblib"))
pred = forest.predict(novel)
proba = forest.predict_proba(novel)

from feature_spec import CLASS_NAMES
counts = np.bincount(pred, minlength=3)
print("Random forest verdict on a fault type it has never seen:")
for c in range(3):
    print(f"  {CLASS_NAMES[c]:<12} {counts[c]:>4} windows  "
          f"(mean confidence {proba[pred == c, c].mean() if counts[c] else 0:.2f})")
print()
print("Note that it is CONFIDENT. It has no mechanism for expressing 'this is")
print("unlike anything I was trained on' -- the softmax must sum to 1 across the")
print("three states it knows. The autoencoder's score above is the missing signal.")

## 7. From anomaly score to Remaining Useful Life

Detection answers *"is something wrong?"*. Maintenance planning needs
*"how long have I got?"* — and the sequence model is most of the way there.

The practical route, roughly in order of effort:

1. **Health Index.** Normalise the reconstruction error against its healthy
   baseline and smooth it. You now have a monotone-ish 0→1 degradation curve,
   which is what the firmware's `degradation_index` approximates with a hand-written
   weighted sum.
2. **Fit a degradation model.** Exponential or Paris-law growth fitted to the
   health index over a sliding horizon, extrapolated to a failure threshold.
   Gives a point estimate of RUL in hours.
3. **Quantify uncertainty.** A point estimate of RUL is nearly useless for
   planning — "fails in 40 hours ± 35" and "fails in 40 hours ± 3" imply totally
   different decisions. A Bayesian filter over the degradation state, or a
   quantile-regression head, gives the interval.
4. **Direct RUL regression.** With enough genuine run-to-failure histories, train
   the LSTM to output RUL directly. This needs *real* data — a synthetic
   generator cannot teach a model when a real bearing dies, and a RUL model
   validated only against its own simulator is circular.

## Why this stays off the microcontroller

It would technically fit. The model above is ~15k parameters, which quantises to
roughly 15 kB — well within the Pico's 2 MB flash and 264 kB RAM. That is not the
reason.

- **No framework, no free lunch.** An LSTM needs real matrix multiplies, which
  means TensorFlow Lite Micro or equivalent: ~20 kB of runtime before your model,
  plus an operator set to maintain. The forest needs *nothing* — it compiles to
  branches.
- **M0+ has no FPU.** Every multiply-accumulate is a software float routine.
  Forest inference is ~120 comparisons; LSTM inference over a 10-step sequence is
  tens of thousands of MACs. The forest answers in microseconds; the LSTM would
  take milliseconds and burn the power budget of a battery node.
- **Thresholds drift, and drift needs retraining.** An anomaly threshold is
  machine-specific and ages as the equipment wears in. Retraining belongs where
  there is history and compute, not on the node.
- **The architecture already has the right place for it.** The node publishes a
  feature vector every 256 ms. The autoencoder runs on that stream at the gateway
  or in the cloud, where it can also compare *across* machines — spotting that
  one of eight identical pumps has started to differ from its siblings, which no
  single node can know.

The honest architecture is therefore **both, at different tiers**: a cheap
supervised classifier at the edge for instant local annunciation and safety
interlocks, and an unsupervised sequence model upstream for novel-fault detection
and RUL. That split is the actual recommendation of this project, and it is why
the firmware's telemetry line carries the whole feature vector rather than just
the verdict.